# B (right): normalized mutual information between course and aligned time

Fills layer `MI` of the figure SVG.

In [1]:
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Helvetica'] + matplotlib.rcParams['font.sans-serif']
matplotlib.rcParams['font.size'] = 6
matplotlib.rcParams['text.usetex'] = False
matplotlib.rcParams["ps.usedistiller"] = 'xpdf'
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.weight'] = 'normal'
matplotlib.rcParams["mathtext.fontset"] = 'cm'

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import entropy as scipy_entropy
from pathlib import Path

import figurefirst as fifi

from align_course_direction_analysis import unifying_algo_analysis as uaa

from scenarios import FIGURE_NAME, SCENARIOS, LOW_SCENARIOS, HIGH_SCENARIOS, SHORT_LABELS, COLORS, load_scenario, save_layer

/Users/florisvb/src/SplitFlowOptoTracking/.venv/lib/python3.11/site-packages/figurefirst/mpl_fig_to_figurefirst_svg.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [3]:
MI_YLIM = (0, 0.2)  # low-range axis, same as the fly figures; the high-range axis expands to fit
RNG_SEED = 0
np.random.seed(RNG_SEED)  # bootstrap reproducibility
fifi_figure_label = 'MI'

In [4]:
def set_scenario_xticks(ax, keys, show_labels=True):
    ax.set_xticks(np.arange(1, len(keys) + 1))
    if show_labels:
        ax.set_xticklabels([SHORT_LABELS[key] for key in keys], rotation=45, ha='right', rotation_mode='anchor')
    else:
        ax.set_xticklabels([])

In [5]:
def analyze_phase_time_structure(data, time_axis=None, n_bins_phase=30, n_bins_time=30,
                                  plot=True, y_range=(-np.pi, np.pi), bootstrap=True,
                                 bootstrap_iterations=1000, bootstrap_subset_fraction=0.8):
    """
    Analyze structure in phase-vs-time data via 2D histogram entropy and mutual information.
    data : (n_trials, n_timepoints) phase values. Returns dict with 'mutual_information', 'H_phase', 'H_time',
    and 'MI_bootstrap' (if bootstrap).
    """
    n_trials, n_timepoints = data.shape

    if time_axis is None:
        time_axis = np.arange(n_timepoints)

    def calc_MI(data, time_axis, n_trials):
        time_flat = np.tile(time_axis, n_trials)
        phase_flat = data.flatten()

        hist2d, phase_edges, time_edges = np.histogram2d(
            phase_flat, time_flat,
            bins=[n_bins_phase, n_bins_time],
            range=[y_range, [time_axis.min(), time_axis.max()]]
        )

        joint = hist2d / hist2d.sum()

        joint_flat = joint.flatten()
        joint_flat = joint_flat[joint_flat > 0]
        H_joint = scipy_entropy(joint_flat)

        p_phase = joint.sum(axis=1)
        p_time  = joint.sum(axis=0)

        H_phase = scipy_entropy(p_phase[p_phase > 0])
        H_time  = scipy_entropy(p_time[p_time > 0])

        MI = H_phase + H_time - H_joint

        return hist2d, phase_edges, time_edges, joint, MI, H_phase, H_time, H_joint

    hist2d, phase_edges, time_edges, joint, MI, H_phase, H_time, H_joint = calc_MI(data, time_axis, n_trials)
    print(f"Observed MI full data set: {MI:.4f}")

    if bootstrap:
        subset_size = max(1, int(bootstrap_subset_fraction*n_trials))
        MI_bootstrap = []
        for i in range(bootstrap_iterations):
            rows = np.random.choice(np.arange(0, n_trials), subset_size, replace=False)
            data_boot = np.vstack([data[row,:] for row in rows])
            _, _, _, _, MIb, _, _, _ = calc_MI(data_boot, time_axis, subset_size)
            MI_bootstrap.append(MIb)

        rows = np.random.choice(np.arange(0, n_trials), subset_size, replace=False)
        data_boot = np.vstack([data[row,:] for row in rows])
        hist2d, phase_edges, time_edges, joint, MI, H_phase, H_time, H_joint = calc_MI(data_boot, time_axis, subset_size)
        print(f"Observed MI boostrap subset size: {MI:.4f}")

    results = {
        'entropy_2d': H_joint,
        'mutual_information': MI,
        'H_phase': H_phase,
        'H_time': H_time,
        'hist2d': joint,
        'phase_edges': phase_edges,
        'time_edges': time_edges,
    }

    if bootstrap:
        results['MI_bootstrap'] = MI_bootstrap

    return results

In [6]:
MIs = {}
for key in SCENARIOS:
    unifying_algo_data, df = load_scenario(key)

    results = uaa.get_aligned_course_array(unifying_algo_data, df, return_linear_and_affine_fits=True)
    ix_master, aligned_time_since_flash_arr, aligned_course_arr, aligned_roi_arr, aligned_linear_fits, aligned_affine_fits = results

    aligned_course_arr_roi = aligned_course_arr[:,200:400]

    print(key)
    result = analyze_phase_time_structure(aligned_course_arr_roi, time_axis=None, n_bins_phase=30, n_bins_time=30,
                                          plot=False, y_range=(-np.pi, np.pi), bootstrap=True)
    MIs[key] = np.array(result['MI_bootstrap']) / min(result['H_phase'], result['H_time'])
    print('   normalized MI: mean %.3f, 95%% CI [%.3f, %.3f]' % (np.mean(MIs[key]), *np.percentile(MIs[key], [2.5, 97.5])))

shuffled_laminar
Observed MI full data set: 0.0947


Observed MI boostrap subset size: 0.1059
   normalized MI: mean 0.031, 95% CI [0.027, 0.035]


shuffled_stillair
Observed MI full data set: 0.1076


Observed MI boostrap subset size: 0.0994
   normalized MI: mean 0.035, 95% CI [0.031, 0.039]


levy_flight
Observed MI full data set: 0.0216


Observed MI boostrap subset size: 0.0229
   normalized MI: mean 0.018, 95% CI [0.016, 0.019]


... (2 outputs left out)

brownian
Observed MI full data set: 0.0278


Observed MI boostrap subset size: 0.0325
   normalized MI: mean 0.009, 95% CI [0.009, 0.010]


run_and_tumble
Observed MI full data set: 0.0303


Observed MI boostrap subset size: 0.0342
   normalized MI: mean 0.010, 95% CI [0.009, 0.011]


In [7]:
## Reference: normalized MI of real flies (flash condition)
# Recomputed here (same window and bootstrap as above) for the 9 flash datasets used in
#   FigureGeneration/Supplemental/sup_fig_unifying_analysis/C_MI_plots_flies_allwtsplitflow.ipynb  (C1xWT flash)
#   FigureGeneration/Supplemental/supp_fig_variable_wind_unifying/D_MI_plots_var_wind.ipynb       (flash, N700)
# Each condition contributes one value (mean of its bootstrap normalized MI, as plotted in those figures).
# The reference line is the median across the 9 conditions; the 95% CI bootstraps over conditions.
FLY_UAA_DIR = Path('../../../Data/unifying_algo_results')
FLY_TRAJEC_DIR = Path('../../../Data/Experimental_Fly_Data')
VARWIND_TRAJEC = FLY_TRAJEC_DIR / 'flies_varwind_C1WT_combined_optotrigger_fancontrol_trimmed.parquet'

FLY_REFERENCE_DATASETS = {
    'laminar':  ('laminar_data_flash_translationTrue.parquet',        FLY_TRAJEC_DIR / 'flies_laminar_c1xwt_preprocessed_optotrigger_trimmed.hdf'),
    'stillair': ('stillair_data_flash_translationTrue.parquet',       FLY_TRAJEC_DIR / 'flies_stillair_c1xwt_preprocessed_optotrigger_trimmed.hdf'),
    'unsteady': ('unsteady_top_on_data_flash_translationTrue.parquet', FLY_TRAJEC_DIR / 'flies_splitflow_topon_c1xwt_preprocessed_optotrigger_trimmed.hdf'),
}
for speed in ['12', '20', '23', '28', '40', '90']:  # fan settings -> 0, 3, 10, 20, 30, 60 cm/s
    FLY_REFERENCE_DATASETS['varwind_' + speed] = ('varwind_N700_flash_' + speed + '.0_translationTrue.parquet', VARWIND_TRAJEC)

N_REFERENCE_BOOTSTRAP = 10000
REFERENCE_COLOR = 'red'

In [8]:
_trajec_cache = {}
def load_fly_trajec(fname):
    if fname not in _trajec_cache:
        _trajec_cache[fname] = pd.read_hdf(fname) if fname.suffix == '.hdf' else pd.read_parquet(fname)
    return _trajec_cache[fname]

fly_MI = {}
for name, (uaa_fname, trajec_fname) in FLY_REFERENCE_DATASETS.items():
    unifying_algo_data = pd.read_parquet(FLY_UAA_DIR / uaa_fname)
    df = load_fly_trajec(trajec_fname)

    results = uaa.get_aligned_course_array(unifying_algo_data, df, return_linear_and_affine_fits=True)
    aligned_course_arr = results[2]

    print(name, 'n =', aligned_course_arr.shape[0])
    result = analyze_phase_time_structure(aligned_course_arr[:,200:400], time_axis=None, n_bins_phase=30, n_bins_time=30,
                                          plot=False, y_range=(-np.pi, np.pi), bootstrap=True)
    MI = np.array(result['MI_bootstrap']) / min(result['H_phase'], result['H_time'])
    fly_MI[name] = np.mean(MI)
    print('   normalized MI: mean %.3f' % fly_MI[name])
_trajec_cache.clear()

fly_values = np.array(list(fly_MI.values()))
rng_ref = np.random.default_rng(RNG_SEED)
boot_medians = np.median(rng_ref.choice(fly_values, size=(N_REFERENCE_BOOTSTRAP, len(fly_values)), replace=True), axis=1)
REFERENCE_MEDIAN = np.median(fly_values)
REFERENCE_CI = np.percentile(boot_medians, [2.5, 97.5])
print('Fly flash reference (%d conditions): median %.3f, 95%% CI [%.3f, %.3f]' % (len(fly_values), REFERENCE_MEDIAN, *REFERENCE_CI))

laminar n = 145
Observed MI full data set: 0.3303


Observed MI boostrap subset size: 0.3546
   normalized MI: mean 0.104


stillair n = 200
Observed MI full data set: 0.3914


Observed MI boostrap subset size: 0.4006
   normalized MI: mean 0.120


unsteady n = 60
Observed MI full data set: 0.3857


Observed MI boostrap subset size: 0.4204
   normalized MI: mean 0.125


... (8 outputs left out)

varwind_40 n = 272
Observed MI full data set: 0.2526


Observed MI boostrap subset size: 0.2546
   normalized MI: mean 0.078


varwind_90 n = 241
Observed MI full data set: 0.2790


Observed MI boostrap subset size: 0.2891
   normalized MI: mean 0.087
Fly flash reference (9 conditions): median 0.104, 95% CI [0.087, 0.125]


In [9]:
# high-range axis: 0.2 as in the fly figures, expanded (to the next 0.1) if a 95% CI exceeds it
mi_max_high = max(np.percentile(MIs[key], 97.5) for key in HIGH_SCENARIOS)
MI_HIGH_YLIM = (0, max(MI_YLIM[1], np.ceil(mi_max_high * 10) / 10))
print('MI_YLIM =', MI_YLIM, ' MI_HIGH_YLIM =', MI_HIGH_YLIM)
for key in LOW_SCENARIOS:
    if np.percentile(MIs[key], 97.5) > MI_YLIM[1]:
        print('   NOTE: %s exceeds the low-range MI axis (95%% CI up to %.3f)' % (key, np.percentile(MIs[key], 97.5)))

def plot_MI(ax, keys, ylim, ylabel=True):
    xixs = np.arange(1, len(keys) + 1)

    # fly flash reference: median (line) and 95% CI (shading) across the 9 fly conditions
    ax.fill_between([xixs[0]-0.5, xixs[-1]+0.5], REFERENCE_CI[0], REFERENCE_CI[1],
                    facecolor=REFERENCE_COLOR, alpha=0.2, edgecolor='none', zorder=0.5)  # > rasterization zorder 0, < bars
    ax.hlines(REFERENCE_MEDIAN, xixs[0]-0.5, xixs[-1]+0.5, colors=REFERENCE_COLOR, linewidths=0.75, zorder=0.6)

    for xix, key in zip(xixs, keys):
        MI = MIs[key]
        ci_low, ci_high = np.percentile(MI, [2.5, 97.5])
        mean = np.mean(MI)
        ax.fill_betweenx([ci_low, ci_high], xix-0.3, xix+0.3, facecolor=COLORS[key], alpha=0.3, edgecolor='none')
        ax.hlines(mean, xix-0.3, xix+0.3, colors=COLORS[key], alpha=1, linewidths=2)

    ax.set_rasterization_zorder(0)
    ax.set_ylim(*ylim)
    yticks = np.round(np.arange(0, ylim[1] + 1e-9, 0.1), 1)
    ax.set_yticks(yticks)
    ax.set_xlim(xixs[0]-0.5, xixs[-1]+0.5)
    ax.set_xticks(xixs)

    ax.tick_params(axis='y', pad=2)
    ax.tick_params(axis='x', pad=2)

    fifi.mpl_functions.adjust_spines(ax, ['left', 'bottom'],
                                     tick_length=2.5,
                                     spine_locations={'left': 5, 'bottom': 5},
                                     linewidth=0.5)
    # tick labels after adjust_spines, which would otherwise reset them
    set_scenario_xticks(ax, keys)
    ax.set_yticklabels(['%g' % yticks[0]] + [''] * (len(yticks) - 2) + ['%g' % yticks[-1]])
    if ylabel:
        ax.set_ylabel('Normalized\nmutual information', labelpad=1)
    fifi.mpl_functions.set_fontsize(ax, 6)

MI_YLIM = (0, 0.2)  MI_HIGH_YLIM = (0, 0.2)


In [10]:
layout = fifi.svg_to_axes.FigureLayout(FIGURE_NAME, autogenlayers=True, make_mplfigures=True, hide_layers=[], dpi=600)
plt.close('all')

plot_MI(layout.axes[(fifi_figure_label, 'MI')], LOW_SCENARIOS, MI_YLIM)
plot_MI(layout.axes[(fifi_figure_label, 'MI_high')], HIGH_SCENARIOS, MI_HIGH_YLIM, ylabel=False)

In [11]:
save_layer(layout, fifi_figure_label)